# 第一章（二）：从音符事件到数组、表格与图形

同一组八个音符事件依次用于数组运算、表格整理、图形绘制和接口核查：

1. 用 NumPy 数组表示数值字段并计算统计量；
2. 用 pandas DataFrame 整理带列名的事件表；
3. 用 Matplotlib 绘制旋律轮廓、音高计数和钢琴卷帘；
4. 查阅当前版本的官方文档，并用简短的对照代码核对函数行为。

数据由同目录的 `note_events.py` 提供，不读取 MIDI 或 MusicXML 文件。所有统计与图形均描述这八个给定事件。

## 1. 环境与配置

先核对 Jupyter 内核实际使用的解释器、cwd 和核心库版本。若这里的 `sys.executable` 与命令行中运行 `00_environment_check.py` 时不同，应重新选择内核，再依次执行 **Restart Kernel** 和 **Run All**。

In [ ]:
import inspect
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"Python：{sys.version.split()[0]}")
print(f"解释器：{sys.executable}")
print(f"当前工作目录：{Path.cwd().resolve()}")
print(f"NumPy：{np.__version__}")
print(f"pandas：{pd.__version__}")
print(f"Matplotlib：{matplotlib.__version__}")

In [ ]:
search_dir = Path.cwd().resolve()

while True:
    if (search_dir / "CODE" / "chapter01").is_dir() and (search_dir / "BOOK").is_dir():
        PROJECT_ROOT = search_dir
        break
    if search_dir.parent == search_dir:
        raise FileNotFoundError("未找到同时包含 CODE/chapter01 和 BOOK 的项目根目录")
    search_dir = search_dir.parent

CHAPTER_DIR = PROJECT_ROOT / "CODE" / "chapter01"
FIGURES_DIR = CHAPTER_DIR / "output_figures"
print(f"本章代码目录：{CHAPTER_DIR}")

if str(CHAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(CHAPTER_DIR))

from note_events import make_note_events

顶部常量控制整个 Notebook。修改移调量、钢琴卷帘的时间分辨率或图像保存开关后，应从头运行。`STEPS_PER_BEAT = 4` 表示每拍分成四个等长时间格；它只决定本例图形的离散网格，不改变原始事件的起始拍和时值。

In [ ]:
TRANSPOSE_SEMITONES = 0
STEPS_PER_BEAT = 4
LONG_NOTE_THRESHOLD_BEATS = 1.0
SAVE_FIGURES = True
FIGURE_DPI = 200

if isinstance(STEPS_PER_BEAT, bool) or not isinstance(STEPS_PER_BEAT, int) or STEPS_PER_BEAT <= 0:
    raise ValueError("STEPS_PER_BEAT 必须为正整数")

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"移调：{TRANSPOSE_SEMITONES:+d} 个半音")
print(f"钢琴卷帘分辨率：每拍 {STEPS_PER_BEAT} 格")
print(f"保存图形：{SAVE_FIGURES}")

`plt.rcParams` 是一个类似字典的全局配置对象。下面通过字符串键设置默认图形尺寸、网格和保存分辨率。示例图使用英文标签，避免依赖本机是否安装中文字体；改用中文标签时，需要显式选择支持相应字符的字体。

In [ ]:
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = FIGURE_DPI
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## 2. 一段短旋律的事件记录

每个事件包含起始位置、MIDI 音高、时值和触发力度。时间单位统一为拍（beat）；`pitch_midi` 取 0—127，`velocity_midi` 取 1—127。触发力度编号常用于控制音源响应，不直接等于声学响度。事件记录不包含速度（tempo）变化、休止、演奏法或踏板信息。

In [ ]:
NOTE_EVENTS = make_note_events(TRANSPOSE_SEMITONES)

if not all(0 <= event["pitch_midi"] <= 127 for event in NOTE_EVENTS):
    raise ValueError("移调后的 MIDI 音高必须在 0 至 127 之间")
if not all(event["duration_beat"] > 0 for event in NOTE_EVENTS):
    raise ValueError("所有音符时值必须大于 0")

melody_end_beat = max(event["onset_beat"] + event["duration_beat"] for event in NOTE_EVENTS)
print(f"事件数：{len(NOTE_EVENTS)}")
print(f"旋律结束位置：{melody_end_beat:.2f} 拍")
print(f"首个事件：{NOTE_EVENTS[0]}")

## 3. NumPy：数组、形状与逐元素运算

`ndarray` 具有固定的轴数和矩形形状，元素按同一种 `dtype` 解释。`np.asarray` 可将 Python 序列转换为数组；输入已经是符合要求的 `ndarray` 时，它通常可以避免不必要的数据复制。`.shape` 给出各轴长度，`.ndim` 给出轴数，`.dtype` 给出元素的数据类型。对这里的一维数组而言，轴 0 沿事件顺序展开。

In [ ]:
onsets = np.asarray([event["onset_beat"] for event in NOTE_EVENTS], dtype=float)
pitches = np.asarray([event["pitch_midi"] for event in NOTE_EVENTS], dtype=np.int16)
durations = np.asarray([event["duration_beat"] for event in NOTE_EVENTS], dtype=float)
velocities = np.asarray([event["velocity_midi"] for event in NOTE_EVENTS], dtype=np.int16)
event_numbers = np.arange(1, len(NOTE_EVENTS) + 1)

print(f"对象类型：{type(pitches).__name__}")
print(f"pitches.shape：{pitches.shape}")
print(f"pitches.ndim：{pitches.ndim}")
print(f"pitches.dtype：{pitches.dtype}")
print(f"首项 pitches[0]：{pitches[0]}")
print(f"末项 pitches[-1]：{pitches[-1]}")
print(f"前三项 pitches[:3]：{pitches[:3]}")
print(f"事件编号 np.arange(...)：{event_numbers}")

数组间的算术通常逐元素进行。相邻 MIDI 音高编号之差可由两个等长切片相减得到。比较运算会产生与原数组形状相同、由 `True` 和 `False` 组成的布尔数组；将它作为索引时，只有对应值为 `True` 的元素会被选出。这种用于选择位置的数组称为布尔掩码（boolean mask）。`std()` 默认使用 `ddof=0`，即以元素个数 $N$ 为除数；这里得到的是对八个给定音高的描述性标准差。

In [ ]:
intervals = pitches[1:] - pitches[:-1]
long_note_mask = durations >= LONG_NOTE_THRESHOLD_BEATS
long_note_pitches = pitches[long_note_mask]

print(f"相邻 MIDI 音高差（半音）：{intervals}")
print(f"平均音高：{pitches.mean():.2f}")
print(f"音高标准差（ddof=0）：{pitches.std():.2f}")
print(f"最低/最高音：{pitches.min()} / {pitches.max()}")
print(f"NumPy 函数得到的最低/最高音：{np.min(pitches)} / {np.max(pitches)}")
print(f"时值总和：{durations.sum():.2f} 拍")
print(f"长音掩码：{long_note_mask}")
print(f"达到阈值的音高：{long_note_pitches}")

`np.arange(start, stop, step)` 按步长生成序列，通常不包含 `stop`；使用非整数步长时应留意浮点舍入。`np.linspace(start, stop, num=...)` 则生成指定数量的等距点，默认包含终点 `stop`，是否包含终点由 `endpoint` 控制。下面只为第一个音符建立五个观察位置；这些位置本身并不定义音频采样率。

In [ ]:
first_note_positions = np.linspace(
    onsets[0],
    onsets[0] + durations[0],
    num=5,
    endpoint=True,
)
print(f"第一个音符区间内的五个等距位置：{first_note_positions}")

把多个一维数组按列组合后，二维数组的轴 0 对应事件，轴 1 对应字段。NumPy 会为各列选择共同的数据类型，因此这个矩阵中的 MIDI 音高和力度也会以浮点数显示；字段的单位与含义并未改变。沿 `axis=0` 聚合会分别对每一列计算，结果仍需按字段解释。

广播（broadcasting）是 NumPy 处理不同形状数组之间逐元素运算的匹配规则。NumPy 从末轴开始向前比较各维度；轴数不同时，较短形状左侧缺少的维度按 1 处理。对应维度相等或其中一个为 1 时可以广播，否则运算失败。长度为 1 的维度通常只在运算逻辑上扩展，不必预先复制成目标形状；运算结果仍需占用与输出形状相应的内存。

`pitches` 的形状是 `(8,)`。写成 `pitches[:, None]` 会增加一个长度为 1 的轴，得到 `(8, 1)`；`octave_offsets` 的形状 `(2,)` 在对齐时视为 `(1, 2)`。运算中，前者的单列在第二个维度扩展为两列，后者的单行在第一个维度扩展为八行，因而得到形状为 `(8, 2)` 的结果。每一行的两个值分别是原音高与在其基础上增加 12 个半音的音高。

In [ ]:
event_matrix = np.column_stack([onsets, pitches, durations, velocities])
column_means = event_matrix.mean(axis=0)
octave_offsets = np.array([0, 12])
pitch_octave_pairs = pitches[:, None] + octave_offsets

print(f"event_matrix.shape：{event_matrix.shape}")
print("沿 axis=0 的列均值 [起始拍, MIDI 音高, 时值拍, 力度]：")
print(column_means)
print(f"广播结果形状：{pitch_octave_pairs.shape}")
print("前三个音高的原位与高八度：")
print(pitch_octave_pairs[:3])

## 4. pandas：带行列标签的事件表

DataFrame 是带行列标签的二维表格，不同列可以使用不同的数据类型。它适合整理字段含义各异的事件记录；NumPy 数组更适合形状规则、数据类型统一的数值运算。两者可以互相转换，转换时仍需保留字段单位和含义。

In [ ]:
events_df = pd.DataFrame(NOTE_EVENTS)
events_df.insert(0, "event_number", np.arange(1, len(events_df) + 1))
events_df["end_beat"] = events_df["onset_beat"] + events_df["duration_beat"]

print(f"DataFrame 形状：{events_df.shape}")
events_df.head(3)

以单个列名索引 DataFrame，结果为 Series；以列名列表索引，结果仍为 DataFrame。对一列进行比较会得到与原表共享行索引的布尔 Series，用它筛选可保留对应值为 `True` 的行。`.to_csv()` 将表格转换为 CSV 格式：提供文件路径时写入文件，不提供路径时返回 CSV 字符串。例如，`events_df.to_csv(index=False)` 返回不含行索引的 CSV 字符串。字段含义、单位与数据来源需通过数据说明或元数据另行记录。

In [ ]:
pitch_column = events_df["pitch_midi"]
timing_table = events_df[["event_number", "onset_beat", "duration_beat", "end_beat"]]
long_events_df = events_df[events_df["duration_beat"] >= LONG_NOTE_THRESHOLD_BEATS]

print("音高列：")
print(pitch_column.to_list())
print("\n时间字段表：")
print(timing_table.to_string(index=False))
print("\n达到时值阈值的事件：")
print(long_events_df.to_string(index=False))

In [ ]:
csv_text = events_df.to_csv(index=False)
print("CSV 前四行：")
print("\n".join(csv_text.splitlines()[:4]))

## 5. Matplotlib：三种图形

Matplotlib 将一幅图组织为 Figure、一个或多个 Axes，以及其中的折线、柱形、文字和图像等绘图元素。`plt.subplots()` 返回 Figure 和 Axes：前者管理整幅图的尺寸、布局与输出，后者包含坐标系统，并提供大部分绘图与标注方法。通过 `ax.plot()`、`ax.set()` 等对象方法显式操作 Axes，在包含多个子图时尤其便于确认修改作用于哪个绘图区。

绘图方法应与数据结构对应：`plot()` 按输入顺序连接成对的横纵坐标，`bar()` 用矩形的高度或宽度表示离散位置的数值，`imshow()` 将二维数组单元映射为颜色。比较多幅图时，还应保持坐标范围、刻度、比例以及颜色归一化方式一致，避免由显示设置制造并不存在于数据中的差异。`tight_layout()` 用于调整图内元素的间距；`savefig()` 中的 `dpi` 影响栅格图像的像素分辨率，`bbox_inches="tight"` 可收紧保存图形外围的空白。

### 5.1 旋律轮廓

横轴是音符起始位置（拍），纵轴是 MIDI 音高编号。连线用于呈现旋律线条的走向，只表示事件之间的顺序，不对应实际演奏中的连续音高变化。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(onsets, pitches, marker="o", linewidth=1.8, color="black")
ax.set(
    title="Melodic contour of eight note events",
    xlabel="Onset (beat)",
    ylabel="MIDI pitch number",
)
ax.set_xlim(0, melody_end_beat)
ax.set_xticks(np.arange(0, int(np.ceil(melody_end_beat)) + 1, 1))
ax.set_yticks(np.arange(pitches.min(), pitches.max() + 1, 2))
fig.tight_layout()

if SAVE_FIGURES:
    contour_path = FIGURES_DIR / "01_melodic_contour.png"
    fig.savefig(contour_path, bbox_inches="tight")
    print(f"已保存：{contour_path}")

plt.show()

### 5.2 音高计数

柱高表示每个 MIDI 音高编号在八个事件中出现的次数。它不按时值加权，也不等同于调性分析。

In [ ]:
unique_pitches, pitch_counts = np.unique(pitches, return_counts=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(unique_pitches, pitch_counts, width=0.7, color="0.45", edgecolor="black")
ax.set(
    title="Pitch counts in eight note events",
    xlabel="MIDI pitch number",
    ylabel="Event count",
)
ax.set_xticks(unique_pitches)
ax.set_yticks(np.arange(0, pitch_counts.max() + 1, 1))
fig.tight_layout()

if SAVE_FIGURES:
    counts_path = FIGURES_DIR / "02_pitch_counts.png"
    fig.savefig(counts_path, bbox_inches="tight")
    print(f"已保存：{counts_path}")

plt.show()

### 5.3 钢琴卷帘矩阵

钢琴卷帘在时间—音高网格上记录音符活动。本例数组的轴 0 对应连续 MIDI 音高编号，轴 1 对应离散时间格，单元格数值保存触发力度。起始位置与终止位置乘以每拍格数后，通过 `round` 转为整数索引；在这组数据中，各起止位置都恰好落在网格边界上，每拍四格时，0.5 拍占两格。若同一音高的事件在某些时间格重叠，下面的代码保留较大的触发力度值。时间离散化会限制可表示的起始位置和时值精度。

绘制矩阵时，`origin="lower"` 将较低音高放在图的下方，`extent` 把数组边界换算为拍和 MIDI 音高编号，`aspect="auto"` 允许绘图区按当前尺寸调整横纵比例，`interpolation="nearest"` 避免在相邻时间格之间插入平滑颜色。`vmin=0` 与 `vmax=127` 固定触发力度的颜色范围，颜色条则给出灰度与数值之间的对应关系。

In [ ]:
total_beats = float(melody_end_beat)
total_steps = int(np.ceil(total_beats * STEPS_PER_BEAT))
lowest_pitch = int(pitches.min())
highest_pitch = int(pitches.max())
piano_roll = np.zeros(
    (highest_pitch - lowest_pitch + 1, total_steps),
    dtype=np.uint8,
)

for onset, pitch_value, duration, velocity in zip(onsets, pitches, durations, velocities):
    start_step = int(round(float(onset) * STEPS_PER_BEAT))
    end_step = int(round(float(onset + duration) * STEPS_PER_BEAT))
    if end_step <= start_step:
        raise ValueError("当前时间分辨率无法表示某个正时值音符")
    pitch_row = int(pitch_value) - lowest_pitch
    piano_roll[pitch_row, start_step:end_step] = np.maximum(
        piano_roll[pitch_row, start_step:end_step],
        velocity,
    )

print(f"钢琴卷帘形状（音高行, 时间列）：{piano_roll.shape}")
print(f"钢琴卷帘数据类型：{piano_roll.dtype}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
image = ax.imshow(
    piano_roll,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    cmap="Greys",
    vmin=0,
    vmax=127,
    extent=[0, total_beats, lowest_pitch - 0.5, highest_pitch + 0.5],
)
ax.set(
    title=f"Piano roll at {STEPS_PER_BEAT} steps per beat",
    xlabel="Time (beat)",
    ylabel="MIDI pitch number",
)
ax.set_xticks(np.arange(0, int(np.ceil(total_beats)) + 1, 1))
ax.set_yticks(np.arange(lowest_pitch, highest_pitch + 1, 2))
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("MIDI velocity")
fig.tight_layout()

if SAVE_FIGURES:
    roll_path = FIGURES_DIR / "03_piano_roll.png"
    fig.savefig(roll_path, bbox_inches="tight")
    print(f"已保存：{roll_path}")

plt.show()

## 6. 读懂调用并核对官方文档

遇到陌生调用时，可从以下几项入手：

1. 从导入语句判断包、模块层次和别名；
2. 区分位置参数、关键字参数、默认值和返回值；
3. 用 `help()`、Notebook 的 `?函数名` 或编辑器悬停查看签名与 docstring；
4. 根据问题选择 Quickstart/Tutorial、User Guide、API Reference、Examples/Gallery 或 Release Notes；
5. 核对文档版本，并构造简短的对照调用；
6. 记录版本、返回类型、形状和关键字段，再判断解释是否与实际结果一致。

以 `np.linspace(0, 1, num=5)` 为例：`np` 是 NumPy 的别名，`linspace` 是模块中的函数，`0` 和 `1` 是位置参数，`num=5` 是关键字参数，返回值是 NumPy 数组。

In [ ]:
print(f"np.linspace 签名：{inspect.signature(np.linspace)}")
print(f"plt.subplots 签名：{inspect.signature(plt.subplots)}")
print(f"DataFrame.head 签名：{inspect.signature(pd.DataFrame.head)}")

下面的 `help()` 读取当前环境中安装版本的 docstring。当前使用的 IPython 内核还支持在独立代码行写 `np.linspace?`；VS Code 中也可悬停查看签名。docstring 便于快速认读，具体版本差异仍应查阅官方页面和发行说明。

In [ ]:
help(np.linspace)

对照实验应保持其他输入不变，只改变待核对的参数。前面的调用使用 `endpoint=True`；这里将其改为 `False`，并打印当前 NumPy 版本、返回类型、形状、数据类型和实际数值。

In [ ]:
test_points = np.linspace(0, 1, num=5, endpoint=False)
print(f"NumPy 版本：{np.__version__}")
print(f"返回类型：{type(test_points)}")
print(f"形状：{test_points.shape}")
print(f"数据类型：{test_points.dtype}")
print(f"实际数值：{test_points}")
assert test_points.shape == (5,)
assert test_points[-1] < 1

以下链接对应第一章锁定的版本系列：

- [`numpy.linspace`](https://numpy.org/doc/2.4/reference/generated/numpy.linspace.html)：`num`、`endpoint` 与返回值；
- [`pandas.DataFrame.head`](https://pandas.pydata.org/pandas-docs/version/3.0/reference/api/pandas.DataFrame.head.html)：默认行数和返回对象；
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/3.10.9/api/_as_gen/matplotlib.pyplot.subplots.html)：Figure、Axes 与子图参数；
- [`matplotlib.axes.Axes.imshow`](https://matplotlib.org/3.10.9/api/_as_gen/matplotlib.axes.Axes.imshow.html)：二维数组、原点、纵横比和颜色映射。

若运行环境中的版本不同，应切换到相应版本的官方页面。AI 助手、搜索摘要、docstring、官方页面与实际输出不一致时，先确认它们是否对应同一软件版本，再以匹配当前环境的官方文档和可重复运行的结果为依据。

## 小结

后续各章会陆续使用 `pretty_midi`、`music21`、`soundfile`、`librosa`、`torch` 等库。无需预先记住全部 API；遇到陌生调用时，先确认解释器和文档版本，再检查输入类型、数组形状、单位和返回对象，通常能够把问题缩小到可验证的范围。